In [ ]:
from ase.io import read
from ase.visualize import view

import nqetools as nqe
import os

import numpy as np
from matplotlib import pyplot as plt

import nqetools as nqe

In [ ]:
# Make a directory to store everything
directory_opti = "opti"
directory_md = "md"
directory_meta_md = "meta_md"
directory_meta_pimd = "meta_pimd"

tol_energy = 5.0e-4
tol_force = 5.0e-4
tol_position = 1.0e-4
total_steps = 1000

n_beads = 4
timestep = 1.0  # fs
total_steps = 5000
total_steps_md = 100

fix_com = True

stride = 10
temperature = 300
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'cbe' #'ase-mace'
driver_args = {}
# # Expensive settings
# driver_args = {'model': 'large',
#                'device': 'cuda',
#                'default_dtype': 'float64'}
#
# # Cheap settings
# driver_args = {'model': 'small',
#                'device': 'cuda',
#                'default_dtype': 'float32'}


In [ ]:
atoms = nqe.read_ipi_xyz("react.xyz")[-1]

In [ ]:
# Run minimisation
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code,
                          tol_energy=tol_energy,
                          tol_force=tol_force,
                          tol_position=tol_position)
atoms_opti, output_data_opti, output_desc_opti = output
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti, save=False)

In [ ]:
view(atoms_opti) # 1, 0, 5

In [ ]:
# Plumed hills settings
n_bins = 100
stride_hills = 100
plumed_type_opes = "opes-diff1"
plumed_args_opes = {'idx1': 1,
                    'idx2': 0,
                    'idx3': 5,
                    'barrier': 0.5,
                    'temperature': temperature,
                    'stride_hills': stride_hills,
                    'explore': False}


plumed_type_opes = "opes-pt2_a"
plumed_args_opes = {'idx1': 0,
                    'idx2': 1,
                    'idx3': 2,
                    'idx4': 3,
                    'barrier': 0.5,
                    'temperature': temperature,
                    'stride_hills': stride_hills,
                    'explore': True}
cv_limits = [None, None]

In [ ]:
# Run unbiased MD
output = nqe.run_md(directory_md,
                    atoms_opti,
                    driver=driver_code,
                    driver_args=driver_args,
                    total_steps=total_steps_md,
                    temperature=temperature,
                    timestep=timestep,
                    thermostat=thermostat,
                    md_type=md_type,
                    fix_com=fix_com,
                    stride=1,
                    n_beads=1)
atoms_md, output_data_md, output_desc_md = output

In [ ]:
# Run OPES metadynamics
output = nqe.run_plumed_md(directory_meta_md,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=1,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_md, output_data_meta_md, output_desc_meta_md = output

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_md, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_md, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_md, save=False)